In [1]:
import gzip
import os
import numpy as np

files = {
    "train_images" : "train-images-idx3-ubyte.gz", 
    "train_labels" : "train-labels-idx1-ubyte.gz", 
    "test_images" : "t10k-images-idx3-ubyte.gz",
    "test_labels" : "t10k-labels-idx1-ubyte.gz"
}

data = {}

datadir = "./data"

for key in ("test_images", "train_images"):
    with gzip.open(os.path.join(datadir, files[key]), "rb") as file: 
        data[key] = np.frombuffer(file.read(), offset = 16, dtype = np.uint8).reshape(-1, 28*28)
        
for key in ("test_labels", "train_labels"): 
    with gzip.open(os.path.join(datadir, files[key]), "rb") as file: 
        data[key] = np.frombuffer(file.read(), offset = 8, dtype = np.uint8)


In [2]:
def leakyRelu(x): 
    return np.where(x > 0, x, x * 0.01)

def softmax(X): 
    
    X = np.exp(X - np.max(X))
    return X / np.sum(X)
    
def leakyReluDeriv(x): 
    return np.where(x > 0, 1. , 0.01 )

def cross_entropy(p , q): 
    
    return -np.dot(p, np.log(q))


In [3]:
def heinit(input_count, output_count): 
    return np.random.normal(0, np.sqrt(2. / input_count), (output_count, input_count))

In [4]:
def one_hot_encode(y): 
    
    r = []
    for i in y: 
        s = np.zeros(10)
        s[i] = 1. 
        r.append(s)
        
    return np.array(r)

In [18]:
def clip_gradient(grad, theresold = 5): 
    norm = np.linalg.norm(grad)
    grad = grad*(theresold/norm)
    return grad

def normalize_grad(grad, theresold = 3): 
    norm = np.linalg.norm(grad)
    if norm > theresold: 
        grad = grad*(theresold/norm)
    return grad
    

In [5]:
data["test_labels"] = one_hot_encode(data["test_labels"])

In [6]:
data["train_labels"] = one_hot_encode(data["train_labels"])

In [115]:
class NeuralNetwork: 
    
    def __init__(self, arch): 
        
        self.arch = arch
        self.layer_count = len(arch)
        
        assert self.layer_count > 2, "not a neural network!"
        
        self.weights = []
        self.not_activated = []
        self.activated = []
        self.biases = []
        self.deltas = []
        
        
        self.weight_updates = []
        self.bias_updates = []
        
        for i in range(self.layer_count - 1): 
            self.weight_updates.append(np.zeros((self.arch[i+1], self.arch[i]), dtype = np.float64))
            self.bias_updates.append(np.zeros(self.arch[i+1],dtype = np.float64))
        
        
    def _init(self): 
        
        # He initialization
        
        self.weights = []
        self.biases = []
        
        for i in range(self.layer_count-1): 
            self.weights.append(heinit(self.arch[i], self.arch[i+1]))
            self.biases.append(np.random.normal(0,0.1,self.arch[i+1]))
            
            
    def _update(self, learning_rate, batch_size): 
        
        # find average of N = batch_size gradients obtained 
        for i in range(self.layer_count - 1): 
            self.weight_updates[i] = self.weight_updates[i]*(1./batch_size)
            self.bias_updates[i] = self.bias_updates[i]*(1./batch_size) 
    
        # update values
        for i in range(self.layer_count - 2, -1, -1):   
            self.weights[i] -= learning_rate*self.weight_updates[i]
            self.biases[i] -= learning_rate*self.bias_updates[i]
            
            # clean cache
            self.weight_updates[i].fill(0.)
            self.bias_updates[i].fill(0.)
        
    
    def forward(self, X): 
        
        result = X / 255. 
        # clean cache
        self.activated = []
        self.not_activated = []
        
        
        for i in range(self.layer_count-2): 
            result = (self.weights[i] @ result) + self.biases[i]
            self.not_activated.append(result)
            result = leakyRelu(result)
            self.activated.append(result)
            
        
        result = softmax(self.weights[self.layer_count - 2] @ result + self.biases[self.layer_count - 2])
        return result
    
    
    def backpropagation(self, X, y, pred): 
        
        X = X / 255.
        delta = pred - y
        self.bias_updates[self.layer_count - 2] += delta
        self.weight_updates[self.layer_count - 2] += np.outer(delta, self.activated[self.layer_count - 3])
        
        for i in range(self.layer_count - 3, 0, -1): 
            delta = np.multiply(self.weights[i+1].T @ delta, leakyReluDeriv(self.not_activated[i]))
            
            # gradient clip
            norm = np.linalg.norm(delta)
            if norm > 5: 
                delta = delta*(5/norm)
            
            self.weight_updates[i] += np.outer(delta, self.activated[i-1])
            self.bias_updates[i] += delta
    
        delta = np.multiply(self.weights[1].T @ delta, leakyReluDeriv(self.not_activated[0]))
        # gradient clip
        norm = np.linalg.norm(delta)
        if norm > 5: 
            delta = delta*(5/norm)
            
        self.weight_updates[0] += np.outer(delta, X)
        self.bias_updates[0] += delta 
        

    def train(self, X, y, batch_size, learning_rate, keepweights = False): 
        
        size = len(X)
        assert size == len(y), "not enough data for training!"
        
        if not keepweights: 
            self._init()
        
        trained = 0
        while trained < size: 
            for i in range(min(batch_size, size - trained)): 
                pred = self.forward(X[trained+i])
                self.backpropagation(X[trained+i], y[trained+i], pred)
               
            self._update(learning_rate,batch_size)
            trained += batch_size
            
    def predict(self, X): 
        
        assert len(self.biases) > 0, "network is not trained or not loaded!"
        return np.argmax(self.forward(X))
    
    
    def save(self, paramdir): 
        
        os.makedirs(paramdir, exist_ok = True)
        
        with open(os.path.join(paramdir, "info.txt"), "wt") as infofile: # architecture header
            s = ",".join([str(i) for i in self.arch])
            infofile.write(s)
            
        for i in range(self.layer_count - 1): 
            np.save(os.path.join(paramdir, f"W{i+1}"), self.weights[i])
            np.save(os.path.join(paramdir, f"b{i+1}"), self.biases[i])
    
        

In [116]:
model = NeuralNetwork([28*28, 100, 50, 30, 10])

In [117]:
from random import randint

In [168]:
def select_random(count):
    indexes = []
    for i in range(count): 
        indexes.append(randint(0,59999))
    x = []
    y = []
    for index in indexes: 
        x.append(data["train_images"][index])
        y.append(data["train_labels"][index])
        
    return np.array(x), np.array(y)

def random_permutations(): 
    
    X, y = [], []
    indexes = np.array([i for i in range(60000)])
    np.random.shuffle(indexes)
    for i in indexes: 
        X.append(data["train_images"][i])
        y.append(data["train_labels"][i])
        
    return np.array(X), np.array(y)
        


In [189]:
X, y = random_permutations()
model.train(X, y, 15, 0.002, True)

In [190]:
c = 0 
for i in range(10000): 
    pred = model.predict(data["test_images"][i])
    if pred == np.argmax(data["test_labels"][i]): 
        c += 1 
        
print("Accuracy: ", c / 10000.)

Accuracy:  0.9612


In [191]:
model.save("./params3") # 96 percent with leaky relu and softmax and gradient normalization!!